# 3. Modelamiento y Pipelines Simplificados

En esta etapa del proyecto, entrenaremos dos modelos supervisados:
1. **Random Forest Classifier**
2. **Logistic Regression**

Para garantizar que el preprocesamiento de los datos sea robusto y libre de errores en producción, construiremos un **Pipeline simplificado de scikit-learn**. Este pipeline empaquetará tanto el escalamiento de variables numéricas como la codificación de variables categóricas, integrando todo en un único objeto listo para predecir.

In [1]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## Carga de Datos

Cargaremos el conjunto de datos limpio (`data/titanic_clean.csv`) y separaremos la variable objetivo (`Survived`) de las características descriptoras (features).

In [2]:
df = pd.read_csv('data/titanic_clean.csv')
print(f"Dimensiones del dataset: {df.shape}")
df.head()

Dimensiones del dataset: (891, 8)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## División de Datos (Train/Test Split)

Dividiremos el dataset en un conjunto de entrenamiento (80%) y uno de prueba (20%) para evaluar el rendimiento general de los modelos.

In [3]:
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Registros en entrenamiento: {X_train.shape[0]}")
print(f"Registros en prueba: {X_test.shape[0]}")

Registros en entrenamiento: 712
Registros en prueba: 179


## Construcción del Preprocesador Automático

Definimos qué columnas numéricas requieren escalamiento estándar (`StandardScaler`) y qué columnas categóricas necesitan codificación One-Hot (`OneHotEncoder`). El `ColumnTransformer` coordinará esto automáticamente.

In [4]:
numerical_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_cols = ['Sex', 'Embarked']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)
print("¡Preprocesador definido con éxito!")

¡Preprocesador definido con éxito!


## Modelo 1: Random Forest Classifier Pipeline

Construiremos el pipeline para Random Forest, lo entrenaremos en el conjunto de entrenamiento y evaluaremos en el conjunto de prueba.

In [5]:
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6))
])

pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Accuracy: {acc_rf:.4f}\n")
print("Reporte de Clasificación (Random Forest):")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.8212

Reporte de Clasificación (Random Forest):
              precision    recall  f1-score   support

           0       0.80      0.95      0.87       110
           1       0.89      0.61      0.72        69

    accuracy                           0.82       179
   macro avg       0.84      0.78      0.80       179
weighted avg       0.83      0.82      0.81       179



## Modelo 2: Logistic Regression Pipeline

Construiremos el pipeline para Regresión Logística, lo entrenaremos y mediremos sus métricas comparativas.

In [6]:
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Logistic Regression Accuracy: {acc_lr:.4f}\n")
print("Reporte de Clasificación (Regresión Logística):")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8045

Reporte de Clasificación (Regresión Logística):
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



## Guardar Modelos de Producción

Exportaremos ambos pipelines entrenados utilizando `joblib` para poder cargarlos y utilizarlos directamente en el servidor web de Flask.

In [ ]:
joblib.dump(pipeline_rf, 'model_rf.joblib')
joblib.dump(pipeline_lr, 'model_lr.joblib')
print("Pipelines guardados exitosamente como 'model_rf.joblib' y 'model_lr.joblib'")